<a href="https://colab.research.google.com/github/team0243/Project_ML/blob/main/Differential_Diagnosis_RCC_UCUT_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Importing libraries**

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, roc_auc_score
from sklearn.preprocessing import MinMaxScaler
import time
from sklearn.metrics import precision_score, recall_score
import warnings

# **Loading and Checking the dataset**


---
This file contains CBC (Complete Blood Count) data of patients, which is used for pathological studies.
It focuses on the classification of Renal Cell Carcinoma (RCC) and Upper Tract Urothelial Carcinoma (UTUC).
This data may be used for developing Machine Learning models for medical analysis.



In [ ]:
url = 'https://github.com/team0243/Project_ML/blob/main/Dataset_RCC_UTUC_ML.xlsx?raw=true' # Replace 'username', 'repository', and 'your_file.xlsx' with the actual path
try:
  df = pd.read_excel(url)
except Exception as e:
  print(f"An error occurred: {e}")
  print("Please ensure the link is correct, the file exists, and the proper permissions are set.")

In [ ]:
# Prints information about a DataFrame
df.info()

- The file is in **Excel (.xlsx)** format.  
- It contains **Age, NLR, PLR, WBC, PLT, NE%, LY%, Lymphocytes, and Neutrophil**.

In [ ]:
df.describe(include="all")

In [ ]:
df.head()

**Checkingfor duplicates and MIssing Value**


In [ ]:
#count duokicates  in DataFrame
print(df.duplicated().sum())

In [ ]:
# Count NaN values in DataFrame
df.isna().sum()

In [ ]:
sns.pairplot(df[['Age ', 'NLR', 'PLR', 'WBC', 'Diagnosis']], hue='Diagnosis')
plt.show()

**Checking the balance between RCC and UTUC**

In [ ]:
df['Diagnosis'].value_counts().plot.bar()

In [ ]:
df['Diagnosis'].value_counts(normalize=True)

**The ratio between RCC and UTUC data is noticeably imbalanced. Therefore, we will address the imbalance issue using over-sampling.**


---



**All Features used to prepare data for Machine Learning Models**

In [ ]:
X = df.drop(columns=['Diagnosis'])  # Independent Variables
y = df['Diagnosis'] # Target or Label to be predicted

In [ ]:
X.head(5)

In [ ]:
X.shape, y.shape

**Over-Resampling**

In [ ]:
# To solve the imbalance problem between categories 0 and 1.
# Apply SMOTE (Synthetic Minority Oversampling Technique) – Oversampling
sm = SMOTE(sampling_strategy = 0.90 ,random_state = 25)
X_resampled, y_resampled = sm.fit_resample(X,y)

In [ ]:
# Creating a DataFrame from the resampled data
resampled_df = pd.DataFrame(X_resampled, columns=X.columns)
resampled_df['Diagnosis'] = y_resampled
# Creating the pairplot
sns.pairplot(resampled_df, hue='Diagnosis')
plt.show()

**Splitting training and testing sets**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size = 0.3, random_state = 25)

In [ ]:
y_train.value_counts(), y_test.value_counts()

In [ ]:
# Plotting histogram of y_train value counts
plt.figure(figsize=(10, 5))
y_train.value_counts().plot(kind='bar', color='skyblue')
plt.title('Histogram of y_train Value Counts')
plt.xlabel('Diagnosis')
plt.ylabel('Frequency')
plt.xticks(rotation=0)
plt.show()
# Plotting histogram of y_test value counts
plt.figure(figsize=(10, 5))
y_test.value_counts().plot(kind='bar', color='salmon')
plt.title('Histogram of y_test Value Counts')
plt.xlabel('Diagnosis')
plt.ylabel('Frequency')
plt.xticks(rotation=0)
plt.show()


**Scaling**

**Scaling all features using StandardScaler.**

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Recursive Feature Elimination using (RFE)

In [ ]:
# RFE using by Random forest cassifier
# Create a Random Forest classifier
model_rf = RandomForestClassifier(random_state=25)

# Define a grid of hyperparameters to tune
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

# Use GridSearchCV to find the best hyperparameters
grid_rf = GridSearchCV(model_rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)
grid_rf.fit(X_train, y_train)

# Print the best hyperparameters found
print("Best Hyperparameters:", grid_rf.best_params_)

# Use the best model found by GridSearchCV
best_model_rf = grid_rf.best_estimator_

# Train the model on the full training set
best_model_rf.fit(X_train, y_train)

In [ ]:
# Feature selection with RFE
n_features_range = [2, 4, 6, 7, 8]

for n_features_to_select in n_features_range:
  rfe = RFE(estimator=RandomForestClassifier(random_state=25, n_estimators=200), n_features_to_select=n_features_to_select)
  X_train_rfe = rfe.fit_transform(X_train, y_train)
  X_test_rfe = rfe.transform(X_test)

  # Retrain the model with selected features.
  model = RandomForestClassifier(random_state=25, n_estimators=200)
  model.fit(X_train_rfe, y_train)
  y_pred_rfe = model.predict(X_test_rfe)

  accuracy_rfe = accuracy_score(y_test, y_pred_rfe)

  print(f"Random Forest Model with {n_features_to_select} Features (RFE)")
  print("-------------------------------------------")
  print(f"Accuracy: {accuracy_rfe}")
  print("Classification Report:")
  print(classification_report(y_test, y_pred_rfe))
  print("-------------------------------------------")


In [ ]:
# RFE using by Decision Tree cassifier
# Create a Decision Tree classifier
model_dt = DecisionTreeClassifier(random_state=25)

# Define a grid of hyperparameters to tune
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Use GridSearchCV to find the best hyperparameters
grid_dt = GridSearchCV(model_dt, param_grid, cv=5, scoring='accuracy')
grid_dt.fit(X_train, y_train)

# Print the best hyperparameters found
print("Best Hyperparameters:", grid_dt.best_params_)

# Use the best model found by GridSearchCV
best_model_dt = grid_dt.best_estimator_

# Train the model on the full training set
best_model_dt.fit(X_train, y_train)


In [ ]:
#Feature selection with RFE
n_features_range = [2, 4, 6, 7, 8]

for n_features_to_select in n_features_range:
  rfe = RFE(estimator=DecisionTreeClassifier(random_state=25), n_features_to_select=n_features_to_select)
  X_train_rfe = rfe.fit_transform(X_train, y_train)
  X_test_rfe = rfe.transform(X_test)

  # Retrain the model with selected features.
  model = DecisionTreeClassifier(random_state=25)
  model.fit(X_train_rfe, y_train)
  y_pred_rfe = model.predict(X_test_rfe)

  accuracy_rfe = accuracy_score(y_test, y_pred_rfe)

  print(f"Decision Tree Model with {n_features_to_select} Features (RFE)")
  print("-------------------------------------------")
  print(f"Accuracy: {accuracy_rfe}")
  print("Classification Report:")
  print(classification_report(y_test, y_pred_rfe))
  print("-------------------------------------------")


In [ ]:
# RFE using by Gradient Boosting classifier
# Create a Gradient Boosting classifier
model_gb = GradientBoostingClassifier(random_state=25)

# Define a grid of hyperparameters to tune
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 4, 5],
    'min_samples_split': [2, 4, 6],
    'min_samples_leaf': [1, 2, 3],
}

# Use GridSearchCV to find the best hyperparameters
grid_gb = GridSearchCV(model_gb, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)
grid_gb.fit(X_train, y_train)

# Print the best hyperparameters found
print("Best Hyperparameters:", grid_gb.best_params_)

# Use the best model found by GridSearchCV
best_model_gb = grid_gb.best_estimator_

# Train the model on the full training set
best_model_gb.fit(X_train, y_train)

In [ ]:
# Feature selection with RFE
n_features_range = [2, 4, 6, 7, 8]

for n_features_to_select in n_features_range:
  rfe = RFE(estimator=GradientBoostingClassifier(random_state=25, n_estimators=200), n_features_to_select=n_features_to_select)
  X_train_rfe = rfe.fit_transform(X_train, y_train)
  X_test_rfe = rfe.transform(X_test)

  # Retrain the model with selected features.
  model = GradientBoostingClassifier(random_state=25, n_estimators=200)
  model.fit(X_train_rfe, y_train)
  y_pred_rfe = model.predict(X_test_rfe)

  accuracy_rfe = accuracy_score(y_test, y_pred_rfe)

  print(f"Gradient Boosting Model with {n_features_to_select} Features (RFE)")
  print("-------------------------------------------")
  print(f"Accuracy: {accuracy_rfe}")
  print("Classification Report:")
  print(classification_report(y_test, y_pred_rfe))
  print("-------------------------------------------")


# Models comparison  and Evaluation Model Performance

In [ ]:
# Models and their names
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=25),
    "Logistic Regression": LogisticRegression(max_iter=200,random_state=25),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Neural Network": MLPClassifier(max_iter=200,random_state=25),
    "Random Forest": RandomForestClassifier(random_state=25),
    "Gradient Boosting": GradientBoostingClassifier(random_state=25)
}

# Dictionary to store ROC data
roc_data = {}

# Iterate through models
for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Handle hyperparameter tuning for specific models
    if model_name == "Decision Tree":
        param_grid_dt = { # Use a separate param_grid for DecisionTreeClassifier
            'criterion': ['gini', 'entropy'],
            'max_depth': [3, 5, 10],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': [None, 'sqrt', 'log2']
        }
        grid_search_dt = GridSearchCV(model, param_grid_dt, cv=5, scoring='accuracy') # Pass the correct param_grid_dt
        grid_search_dt.fit(X_train, y_train)
        model = grid_search_dt.best_estimator_
    elif model_name == "Logistic Regression":
        param_grid_ls = { # Corrected parameter grid for Logistic Regression
            'penalty': ['l1', 'l2'],
            'C': [0.001, 0.01, 0.1, 1, 10, 100],
            'solver': ['liblinear', 'saga']
        }
        grid_search_ls = GridSearchCV(model, param_grid_ls, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)
        grid_search_ls.fit(X_train, y_train)
        model = grid_search_ls.best_estimator_
    elif model_name == "K-Nearest Neighbors":
        param_grid_knn = {
            'n_neighbors': [5, 7, 9, 11, 15],
            'weights': ['uniform'],
            'p': [1, 2, 3]
        }
        grid_search_knn = GridSearchCV(model, param_grid_knn, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)
        grid_search_knn.fit(X_train, y_train)
        model = grid_search_knn.best_estimator_
    elif model_name == "Neural Network":
        param_grid_mlp = {
            'hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 50)],
            'activation': ['tanh', 'relu'],
            'solver': ['sgd', 'adam'],
            'alpha': [0.0001, 0.05],
            'learning_rate': ['constant','adaptive']
        }
        grid_search_mlp = GridSearchCV(model, param_grid_mlp, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)
        grid_search_mlp.fit(X_train, y_train)
        model = grid_search_mlp.best_estimator_
    elif model_name == "Random Forest":
        param_grid_rf = {
            'n_estimators': [50, 100, 200],
            'max_depth': [None, 10, 20, 30],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2']
        }
        grid_search_rf = GridSearchCV(model, param_grid_rf, cv=5, scoring='accuracy', n_jobs=-1, verbose=0) # Pass the correct param_grid_rf
        grid_search_rf.fit(X_train, y_train)
        model = grid_search_rf.best_estimator_

    elif model_name == "Gradient Boosting":
        param_grid_gb = {
            'n_estimators': [50 , 100, 150],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [2, 3 ,4],
            'min_samples_split': [4, 6, 8],
            'min_samples_leaf': [2, 3, 4]
        }
        grid_search_gb = GridSearchCV(model, param_grid_gb, cv=5, scoring='accuracy', verbose=0) # Pass the correct param_grid_gb
        grid_search_gb.fit(X_train, y_train)
        model = grid_search_gb.best_estimator_

    # Train the model
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_test_numeric = y_test.map({'RCC': 0, 'UTUC': 1})
    fpr, tpr, thresholds = roc_curve(y_test_numeric, y_prob)
    roc_auc = roc_auc_score(y_test_numeric, y_prob)
    roc_data[model_name] = (fpr, tpr, roc_auc)

# Plotting the ROC curves
plt.figure(figsize=(10, 8))
for model_name, (fpr, tpr, roc_auc) in roc_data.items():
    plt.plot(fpr, tpr, lw=2, label=f'{model_name} (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curves')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Evaluate Model Performance
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=25),
    "Logistic Regression": LogisticRegression(max_iter=200, random_state=25),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Neural Network": MLPClassifier(max_iter=200, random_state=25),
    "Random Forest": RandomForestClassifier(random_state=25),
    "Gradient Boosting": GradientBoostingClassifier(random_state=25)
}

# Parameter grids for hyperparameter tuning
param_grids = {
    "Decision Tree": {
        'criterion': ['gini', 'entropy'],
        'max_depth': [3, 5, 10],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': [None, 'sqrt', 'log2']
    },
    "Logistic Regression": {
        'penalty': ['l1', 'l2'],
        'C': [0.001, 0.01, 0.1, 1, 10, 100],
        'solver': ['liblinear', 'saga']
    },
    "K-Nearest Neighbors": {
        'n_neighbors': [ 5, 7, 9, 11, 15],
        'weights': ['uniform'],
        'p': [1, 2, 3]
    },
    "Neural Network": {
        'hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 50)],
        'activation': ['tanh', 'relu'],
        'solver': ['sgd', 'adam'],
        'alpha': [0.0001, 0.05],
        'learning_rate': ['constant', 'adaptive']
    },
    "Random Forest": {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2']
    },
    "Gradient Boosting": {
        'n_estimators': [50, 100, 150],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [2, 3, 4 ],
        'min_samples_split': [4, 6, 8],
        'min_samples_leaf': [2, 3, 4]
    }
}

# Data structure to store results
results = []

# Train and evaluate models
for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Hyperparameter tuning
    if model_name in param_grids:
        start_time = time.time()
        grid_search = GridSearchCV(model, param_grids[model_name], cv=5, scoring='accuracy', n_jobs=-1, verbose=0)
        grid_search.fit(X_train, y_train)
        model = grid_search.best_estimator_
        training_time = time.time() - start_time
    else:
        start_time = time.time()
        model.fit(X_train, y_train)
        training_time = time.time() - start_time

    # Training score
    training_score = model.score(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)

    # Evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=1)
    recall = recall_score(y_test, y_pred, average='weighted')

    # Store results
    results.append({
        "Model": model_name,
        "Training score": training_score,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "Training time": training_time
    })

# Create results dataframe
results_df = pd.DataFrame(results)
results_df
